In [2]:
from ollama import chat, Client

"""
Configure instruction selection, LLM parameters, API access, and automatic recovery settings for specification processing.
"""

### CHANGE HERE ###
LIST_INSTRUCTION_WANTED = [
    'SFENCE.VMA',
    'CSRRWI',
    'VLE32FF.V',
    'FCVT.W.D',
]
IS_USING_EVERYTHING = False # Set this to True if you want to use all possible instructions
### CHANGE HERE ###

MODEL = 'deepseek-v4.1-flash:1017'
IS_LOCAL = False
TEMPERATURE = 1
CONTEXT = 65536
API_KEY = ''
CLIENT = Client(host='https://ollama.com', timeout=1800, headers={'Authorization': API_KEY})
WIDTH = 120

RESET_PROGRESS = True

# Automatic LLM recovery.
AUTO_RETRY = True
CHECKPOINT_EVERY = 10
MAX_RETRIES = 10
RETRY_WAIT_SECONDS = 30
RUNNER_STOP_TIMEOUT_SECONDS = 60

print('====\nDONE\n====')

====
DONE
====


In [3]:
# 1 - CREATE - 1_INSTRUCTION_ALL.txt - START

"""
Extract and normalize RISC-V instruction mnemonics and assembly formats using riscv-unified-db and riscv-opcodes.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import glob, os, re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '2_INPUT/riscv-unified-db/tests/golden/all_instructions.golden.adoc'
FOLDER_INPUT = '2_INPUT/riscv-opcodes/extensions'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/1_INSTRUCTION_ALL.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_BLOCK_SORT_KEY(STRING_BLOCK: str):
    MATCH = re.search(r'^REGISTER:\s*([^\n]+)', STRING_BLOCK, flags=re.M)
    STRING_NAME = MATCH.group(1).strip() if MATCH else STRING_BLOCK
    return FUNCTION_NATURAL_SORT_KEY(STRING_NAME)

def FUNCTION_NORMALIZE_SUFFIX_FORM(STRING_TOKEN):
    for EACH_PATTERN_SUFFIX, EACH_STRING_KIND in ((r'([a-z_]*imm)(\d+)(?:sp)?(hi|lo)', None), (r'([a-z_]*imm)(\d+)sp(?:_s)?', 'sp'), (r'([a-z_]*imm)(\d+)(hi|lo)', None)):
        MATCH_SUFFIX = re.fullmatch(EACH_PATTERN_SUFFIX, STRING_TOKEN, re.I)
        if MATCH_SUFFIX:
            return (f'{MATCH_SUFFIX.group(1)}{MATCH_SUFFIX.group(2)}', (EACH_STRING_KIND or MATCH_SUFFIX.group(3).lower()))
    if re.fullmatch(r'[a-z_]*imm\d+$', STRING_TOKEN, re.I):
        return STRING_TOKEN, 'plain'
    return None, None

def FUNCTION_CHOOSE_IMM(LIST_OPERAND):
    LIST_IMM = [EACH_TOKEN for EACH_TOKEN in LIST_OPERAND if REGEX_TOKEN_IMM.fullmatch(EACH_TOKEN)]
    if not LIST_IMM:
        return None
    MAP_GROUP = {}
    for EACH_TOKEN in LIST_IMM:
        STRING_BASE, STRING_KIND = FUNCTION_NORMALIZE_SUFFIX_FORM(EACH_TOKEN)
        if STRING_BASE:
            MAP_GROUP.setdefault(STRING_BASE, set()).add(STRING_KIND)
    for EACH_STRING_PREFERENCE in ('hi lo', 'sp', 'plain'):
        for EACH_STRING_BASE, EACH_SET_KIND in MAP_GROUP.items():
            if all(EACH_PREF in EACH_SET_KIND for EACH_PREF in EACH_STRING_PREFERENCE.split()):
                return EACH_STRING_BASE
    STRING_BASE, _ = FUNCTION_NORMALIZE_SUFFIX_FORM(LIST_IMM[0])
    return STRING_BASE or LIST_IMM[0]

def FUNCTION_COLLECT_OPERAND_ANYWHERE(LIST_TOKEN):
    return [EACH_TOKEN for EACH_TOKEN in LIST_TOKEN if not REGEX_TOKEN_ENCODING.match(EACH_TOKEN) and '=' not in EACH_TOKEN and '..' not in EACH_TOKEN and '::' not in EACH_TOKEN]

def FUNCTION_PROMPT(MAP_IMM, STRING_MNEMONIC, LIST_OPERAND):
    STRING_IMM = FUNCTION_CHOOSE_IMM(LIST_OPERAND)
    if STRING_IMM and STRING_MNEMONIC not in MAP_IMM:
        MAP_IMM[STRING_MNEMONIC] = STRING_IMM
# FUNCTION - FINISH

# VARIABLE - BEGIN
REGEX_TOKEN_IMM = re.compile(r'(?:[a-z_]*imm\d+(?:hi|lo|sphi|splo|sp(?:_s)?)?|zimm(?:\d+)?|shamt\d+)$', re.I)
REGEX_TOKEN_ENCODING = re.compile(r'(?:^\d+\.\.\d+=|^\d+=|^0x[0-9a-f]+$|^[01]+$|^\d+\.\.\d+$)', re.I)
MAP_IMM = {}
LIST_BLOCK_FINAL = []
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_LINE = FILE_HANDLE.readlines()

LIST_BLOCK = [(f"INSTRUCTION: {EACH_LINE_NEXT.split()[0].upper()}\nFORMAT: " f"{EACH_LINE_NEXT.strip().replace('xd','rd').replace('xs1','rs1').replace('xs2','rs2')}") for EACH_LINE_CURRENT, EACH_LINE_NEXT in zip(LIST_LINE, LIST_LINE[1:]) if EACH_LINE_CURRENT.startswith('Assembly::')]

for EACH_FILE_PATH in (EACH_PATH for EACH_PATH in glob.glob(os.path.join(FOLDER_INPUT, 'rv*')) if os.path.isfile(EACH_PATH)):
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        for EACH_RAW_LINE in FILE_HANDLE:
            STRING_LINE = EACH_RAW_LINE.strip()
            if not STRING_LINE:
                continue
            if '->' in STRING_LINE:
                try:
                    _, STRING_RIGHT = STRING_LINE.split('->', 1)
                    LIST_TOKEN = STRING_RIGHT.split()
                    if LIST_TOKEN:
                        FUNCTION_PROMPT(MAP_IMM, LIST_TOKEN[0].split(':')[-1], FUNCTION_COLLECT_OPERAND_ANYWHERE(LIST_TOKEN[1:]))
                except ValueError:
                    pass
                continue
            LIST_TOKEN = STRING_LINE.split()
            if not LIST_TOKEN:
                continue
            if LIST_TOKEN[0] == '$pseudo_op' and len(LIST_TOKEN) >= 3:
                FUNCTION_PROMPT(MAP_IMM, LIST_TOKEN[2], FUNCTION_COLLECT_OPERAND_ANYWHERE(LIST_TOKEN[3:]))
                continue
            FUNCTION_PROMPT(MAP_IMM, LIST_TOKEN[0].split(':')[-1], FUNCTION_COLLECT_OPERAND_ANYWHERE(LIST_TOKEN[1:]))

for EACH_STRING_BLOCK in LIST_BLOCK:
    STRING_INSTRUCTION = EACH_STRING_BLOCK.splitlines()[0].split(':', 1)[1].strip()
    STRING_FORMAT = EACH_STRING_BLOCK.splitlines()[1].split(':', 1)[1].strip()
    STRING_IMM = MAP_IMM.get(STRING_INSTRUCTION.lower())
    if STRING_IMM:
        STRING_FORMAT = re.sub(r'\bimm\b(?=\()', STRING_IMM, STRING_FORMAT, flags=re.I)
        STRING_FORMAT = re.sub(r'\bimm\b(?!\()', STRING_IMM, STRING_FORMAT, flags=re.I)
    for EACH_PATTERN_LABEL in (r'\bbimm\d+(hi|lo)?\b', r'\bjimm\d+\b', r'\bc_bimm\d+(hi|lo)?\b', r'\bc_imm12\b'):
        STRING_FORMAT = re.sub(EACH_PATTERN_LABEL, 'label', STRING_FORMAT, flags=re.I)
    if STRING_INSTRUCTION.upper() in ('AUIPC', 'LUI'):
        STRING_FORMAT = STRING_FORMAT.replace('imm20', 'uimm20')
    LIST_BLOCK_FINAL.append(f'INSTRUCTION: {STRING_INSTRUCTION.upper()}\nFORMAT: {STRING_FORMAT}')

LIST_BLOCK_FINAL = [EACH_BLOCK.replace('ssamoswap.d rd, rs2, rs1', 'ssamoswap.d rd, rs2, (rs1)').replace('ssamoswap.w rd, rs2, rs1', 'ssamoswap.w rd, rs2, (rs1)').replace('fcvt.d.h fd, fs1, rm', 'fcvt.d.h fd, fs1') for EACH_BLOCK in LIST_BLOCK_FINAL]

LIST_BLOCK_FINAL.sort(key=FUNCTION_BLOCK_SORT_KEY)

LIST_BLOCK_FINAL = [re.sub(r'^FORMAT:.*$', 'FORMAT: -', EACH_BLOCK, flags=re.M) for EACH_BLOCK in LIST_BLOCK_FINAL]

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n\n'.join(LIST_BLOCK_FINAL))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 1 - CREATE - 1_INSTRUCTION_ALL.txt - END

0.02 second(s)

====
DONE
====


In [4]:
# 2 - CREATE - 2_INSTRUCTION_SOME.txt - START

"""
Select RISC-V instructions and their assembly formats based on user configuration, excluding instructions not yet supported by the framework.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '4_INSTRUCTION/1_OUTPUT/1_INSTRUCTION_ALL.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/2_INSTRUCTION_SOME.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]
# FUNCTION - FINISH

# VARIABLE - BEGIN
LIST_NOT_IMPLEMENTED = ['AES32DSI', 'AES32DSMI', 'AES32ESI', 'AES32ESMI', 'AES64DS', 'AES64DSM', 'AES64ES', 'AES64ESM', 'AES64IM', 'AES64KS1I', 'AES64KS2', 'AMOADD.B', 'AMOADD.B.AQ', 'AMOADD.B.AQRL', 'AMOADD.B.RL', 'AMOADD.D', 'AMOADD.D.AQ', 'AMOADD.D.AQRL', 'AMOADD.D.RL', 'AMOADD.H', 'AMOADD.H.AQ', 'AMOADD.H.AQRL', 'AMOADD.H.RL', 'AMOADD.W', 'AMOADD.W.AQ', 'AMOADD.W.AQRL', 'AMOADD.W.RL', 'AMOAND.B', 'AMOAND.B.AQ', 'AMOAND.B.AQRL', 'AMOAND.B.RL', 'AMOAND.D', 'AMOAND.D.AQ', 'AMOAND.D.AQRL', 'AMOAND.D.RL', 'AMOAND.H', 'AMOAND.H.AQ', 'AMOAND.H.AQRL', 'AMOAND.H.RL', 'AMOAND.W', 'AMOAND.W.AQ', 'AMOAND.W.AQRL', 'AMOAND.W.RL', 'AMOCAS.B', 'AMOCAS.B.AQ', 'AMOCAS.B.AQRL', 'AMOCAS.B.RL', 'AMOCAS.D', 'AMOCAS.D.AQ', 'AMOCAS.D.AQRL', 'AMOCAS.D.RL', 'AMOCAS.H', 'AMOCAS.H.AQ', 'AMOCAS.H.AQRL', 'AMOCAS.H.RL', 'AMOCAS.Q', 'AMOCAS.Q.AQ', 'AMOCAS.Q.AQRL', 'AMOCAS.Q.RL', 'AMOCAS.W', 'AMOCAS.W.AQ', 'AMOCAS.W.AQRL', 'AMOCAS.W.RL', 'AMOMAX.B', 'AMOMAX.B.AQ', 'AMOMAX.B.AQRL', 'AMOMAX.B.RL', 'AMOMAX.D', 'AMOMAX.D.AQ', 'AMOMAX.D.AQRL', 'AMOMAX.D.RL', 'AMOMAX.H', 'AMOMAX.H.AQ', 'AMOMAX.H.AQRL', 'AMOMAX.H.RL', 'AMOMAX.W', 'AMOMAX.W.AQ', 'AMOMAX.W.AQRL', 'AMOMAX.W.RL', 'AMOMAXU.B', 'AMOMAXU.B.AQ', 'AMOMAXU.B.AQRL', 'AMOMAXU.B.RL', 'AMOMAXU.D', 'AMOMAXU.D.AQ', 'AMOMAXU.D.AQRL', 'AMOMAXU.D.RL', 'AMOMAXU.H', 'AMOMAXU.H.AQ', 'AMOMAXU.H.AQRL', 'AMOMAXU.H.RL', 'AMOMAXU.W', 'AMOMAXU.W.AQ', 'AMOMAXU.W.AQRL', 'AMOMAXU.W.RL', 'AMOMIN.B', 'AMOMIN.B.AQ', 'AMOMIN.B.AQRL', 'AMOMIN.B.RL', 'AMOMIN.D', 'AMOMIN.D.AQ', 'AMOMIN.D.AQRL', 'AMOMIN.D.RL', 'AMOMIN.H', 'AMOMIN.H.AQ', 'AMOMIN.H.AQRL', 'AMOMIN.H.RL', 'AMOMIN.W', 'AMOMIN.W.AQ', 'AMOMIN.W.AQRL', 'AMOMIN.W.RL', 'AMOMINU.B', 'AMOMINU.B.AQ', 'AMOMINU.B.AQRL', 'AMOMINU.B.RL', 'AMOMINU.D', 'AMOMINU.D.AQ', 'AMOMINU.D.AQRL', 'AMOMINU.D.RL', 'AMOMINU.H', 'AMOMINU.H.AQ', 'AMOMINU.H.AQRL', 'AMOMINU.H.RL', 'AMOMINU.W', 'AMOMINU.W.AQ', 'AMOMINU.W.AQRL', 'AMOMINU.W.RL', 'AMOOR.B', 'AMOOR.B.AQ', 'AMOOR.B.AQRL', 'AMOOR.B.RL', 'AMOOR.D', 'AMOOR.D.AQ', 'AMOOR.D.AQRL', 'AMOOR.D.RL', 'AMOOR.H', 'AMOOR.H.AQ', 'AMOOR.H.AQRL', 'AMOOR.H.RL', 'AMOOR.W', 'AMOOR.W.AQ', 'AMOOR.W.AQRL', 'AMOOR.W.RL', 'AMOSWAP.B', 'AMOSWAP.B.AQ', 'AMOSWAP.B.AQRL', 'AMOSWAP.B.RL', 'AMOSWAP.D', 'AMOSWAP.D.AQ', 'AMOSWAP.D.AQRL', 'AMOSWAP.D.RL', 'AMOSWAP.H', 'AMOSWAP.H.AQ', 'AMOSWAP.H.AQRL', 'AMOSWAP.H.RL', 'AMOSWAP.W', 'AMOSWAP.W.AQ', 'AMOSWAP.W.AQRL', 'AMOSWAP.W.RL', 'AMOXOR.B', 'AMOXOR.B.AQ', 'AMOXOR.B.AQRL', 'AMOXOR.B.RL', 'AMOXOR.D', 'AMOXOR.D.AQ', 'AMOXOR.D.AQRL', 'AMOXOR.D.RL', 'AMOXOR.H', 'AMOXOR.H.AQ', 'AMOXOR.H.AQRL', 'AMOXOR.H.RL', 'AMOXOR.W', 'AMOXOR.W.AQ', 'AMOXOR.W.AQRL', 'AMOXOR.W.RL', 'WFI', 'ECALL', 'EBREAK']
# VARIABLE - FINISH

# CODE - BEGIN
SET_NOT_IMPLEMENTED = {EACH_INSTRUCTION.upper() for EACH_INSTRUCTION in LIST_NOT_IMPLEMENTED}

LIST_INSTRUCTION_WANTED = [
    EACH_INSTRUCTION
    for EACH_INSTRUCTION in LIST_INSTRUCTION_WANTED
    if EACH_INSTRUCTION.upper() not in SET_NOT_IMPLEMENTED
]

if not LIST_INSTRUCTION_WANTED:
    LIST_INSTRUCTION_WANTED.append('ADD')

LIST_INSTRUCTION_WANTED.sort(key=FUNCTION_NATURAL_SORT_KEY)

with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_LINE = FILE_HANDLE.readlines()

if IS_USING_EVERYTHING:
    LIST_OUT = []
    EACH_INDEX_LINE = 0
    while EACH_INDEX_LINE < len(LIST_LINE):
        EACH_LINE = LIST_LINE[EACH_INDEX_LINE]
        STRIPPED = EACH_LINE.strip()
        if STRIPPED.startswith('INSTRUCTION:'):
            NAME = STRIPPED.split(':', 1)[1].strip().upper()
            if NAME in SET_NOT_IMPLEMENTED:
                EACH_INDEX_LINE += 1
                while (
                    EACH_INDEX_LINE < len(LIST_LINE)
                    and not LIST_LINE[EACH_INDEX_LINE].strip().startswith('INSTRUCTION:')
                ):
                    EACH_INDEX_LINE += 1
                continue
        LIST_OUT.append(EACH_LINE)
        EACH_INDEX_LINE += 1
    STRING_CONTENT = ''.join(LIST_OUT)
    STRING_CONTENT = re.sub(r'\n{3,}', '\n\n', STRING_CONTENT).strip()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_CONTENT)
else:
    LIST_WANTED = sorted({EACH_INSTRUCTION.upper() for EACH_INSTRUCTION in LIST_INSTRUCTION_WANTED}, key=FUNCTION_NATURAL_SORT_KEY)
    LIST_SELECTED = []
    for EACH_STRING_WANTED in LIST_WANTED:
        for EACH_INDEX_LINE, EACH_STRING_LINE in enumerate(LIST_LINE):
            if EACH_STRING_LINE.strip() == f'INSTRUCTION: {EACH_STRING_WANTED}':
                LIST_SELECTED.append(f'{EACH_STRING_LINE.strip()}\n{LIST_LINE[EACH_INDEX_LINE + 1].strip()}')
                break
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write('\n\n'.join(LIST_SELECTED))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 2 - CREATE - 2_INSTRUCTION_SOME.txt - END

0.00 second(s)

====
DONE
====


In [5]:
# 3 - CREATE - 3_INSTRUCTION_BOOLEAN.txt - START

"""
Identify relevant specification sections for each selected RISC-V instruction using keyword matching and LLM-based relevance classification.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client, ResponseError
import hashlib, json, os, re, subprocess
from pathlib import Path
# IMPORT - FINISH

# PATH - BEGIN
FOLDER_INPUT = '3_SPECIFICATION/2_SPLIT'
FILE_INPUT = '4_INSTRUCTION/1_OUTPUT/2_INSTRUCTION_SOME.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/3_INSTRUCTION_BOOLEAN.txt'
FILE_PROMPT = '4_INSTRUCTION/2_HISTORY/1_HISTORY_INSTRUCTION_BOOLEAN.txt'
FILE_CHECKPOINT = '4_INSTRUCTION/3_WHICH_LOOP/WHICH_LOOP_BOOLEAN.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_NATURAL_SORT_KEY(STRING_VALUE):
    return [int(EACH_TOKEN) if EACH_TOKEN.isdigit() else EACH_TOKEN.lower() for EACH_TOKEN in re.split(r'(\d+)', STRING_VALUE)]

def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+').replace("’", "'").replace("‘", "'"))

def FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR):
    if isinstance(ERROR, ConnectionError):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    STRING_ERROR = str(ERROR).lower()
    LIST_RETRY_KEYWORD = ['cuda', 'gpu', 'launch timed out', 'ggml_backend_cuda', 'cudastreamsynchronize', 'cannot access memory', 'runner']
    if STATUS_CODE is not None and STATUS_CODE >= 500:
        return True
    return any(EACH_KEYWORD in STRING_ERROR for EACH_KEYWORD in LIST_RETRY_KEYWORD)

def FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR):
    if isinstance(ERROR, ConnectionError):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    return STATUS_CODE in (408, 429) or (STATUS_CODE is not None and STATUS_CODE >= 500)

def FUNCTION_STOP_LOCAL_RUNNER(MODEL):
    print(f'[RECOVERY] Stopping Ollama runner for {MODEL!r}...', flush=True)
    try:
        COMPLETED_PROCESS = subprocess.run(['ollama', 'stop', MODEL], check=False, capture_output=True, text=True, timeout=RUNNER_STOP_TIMEOUT_SECONDS)
        if COMPLETED_PROCESS.returncode != 0:
            STRING_ERROR = COMPLETED_PROCESS.stderr.strip()
            if STRING_ERROR:
                print(f'[RECOVERY] ollama stop returned {COMPLETED_PROCESS.returncode}: {STRING_ERROR}', flush=True)
    except Exception as ERROR:
        print(f'[RECOVERY] Could not stop runner cleanly: {ERROR}', flush=True)

def FUNCTION_ASK(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT, TEMPERATURE, CONTEXT):
    NUMBER_ATTEMPT = 0
    while True:
        NUMBER_ATTEMPT += 1
        try:
            if IS_LOCAL:
                return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content.strip()
            return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content.strip()
        except (ResponseError, ConnectionError) as ERROR:
            IS_RETRYABLE = FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR) if IS_LOCAL else FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR)
            if not AUTO_RETRY or not IS_RETRYABLE or NUMBER_ATTEMPT >= MAX_RETRIES:
                print(f'[RECOVERY] Giving up after attempt {NUMBER_ATTEMPT}/{MAX_RETRIES}.', flush=True)
                raise
            STRING_MODE = 'LOCAL' if IS_LOCAL else 'ONLINE'
            print(f'[RECOVERY] {STRING_MODE} request failed on attempt {NUMBER_ATTEMPT}/{MAX_RETRIES}: {ERROR}', flush=True)
            if IS_LOCAL:
                FUNCTION_STOP_LOCAL_RUNNER(MODEL)
            print(f'[RECOVERY] Waiting {RETRY_WAIT_SECONDS} seconds, then retrying the SAME request automatically...', flush=True)
            time.sleep(RETRY_WAIT_SECONDS)

def FUNCTION_WRAP_HISTORY(WIDTH, GLOBAL_NUMBER, LOCAL_NUMBER, STRING_NAME, STRING_PROMPT, STRING_RESPONSE):
    STRING_INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * (WIDTH // 2)}
=======
PROMPT:
=======
{'=' * (WIDTH // 2)}
'''
    STRING_ENDING = f'''
{'=' * (WIDTH // 2)}
=========
RESPONSE:
=========
{'=' * (WIDTH // 2)}
{STRING_RESPONSE}
'''
    return f"{f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')}\n{STRING_INTRO}{STRING_PROMPT}{STRING_ENDING}{f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')}"

def FUNCTION_ATOMIC_WRITE_JSON(FILE_PATH, OBJECT_VALUE):
    FILE_PATH = Path(FILE_PATH)
    FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
    FILE_TEMP = FILE_PATH.with_suffix(FILE_PATH.suffix + '.tmp')
    with open(FILE_TEMP, 'w', encoding='utf-8') as FILE_HANDLE:
        json.dump(OBJECT_VALUE, FILE_HANDLE, indent=2, sort_keys=True)
        FILE_HANDLE.write('\n')
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())
    os.replace(FILE_TEMP, FILE_PATH)

def FUNCTION_APPEND_AND_SYNC(FILE_PATH, STRING_VALUE):
    if not STRING_VALUE:
        return
    with open(FILE_PATH, 'a', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_VALUE)
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())

def FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PATH, COMMITTED_SIZE):
    FILE_PATH = Path(FILE_PATH)
    FILE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not FILE_PATH.exists():
        if COMMITTED_SIZE != 0:
            raise RuntimeError(f'Cannot resume safely: {FILE_PATH} is missing, but checkpoint expects {COMMITTED_SIZE} committed bytes.')
        FILE_PATH.touch()
        return
    CURRENT_SIZE = FILE_PATH.stat().st_size
    if CURRENT_SIZE < COMMITTED_SIZE:
        raise RuntimeError(f'Cannot resume safely: {FILE_PATH} is only {CURRENT_SIZE} bytes, but checkpoint expects {COMMITTED_SIZE} committed bytes.')
    if CURRENT_SIZE > COMMITTED_SIZE:
        with open(FILE_PATH, 'r+b') as FILE_HANDLE:
            FILE_HANDLE.truncate(COMMITTED_SIZE)
            FILE_HANDLE.flush()
            os.fsync(FILE_HANDLE.fileno())

def FUNCTION_BUILD_WORK_ITEMS():
    with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
        LIST_INSTRUCTION = [EACH_LINE.split(': ')[1].strip() for EACH_LINE in FILE_HANDLE if EACH_LINE.startswith('INSTRUCTION: ')]
    LIST_FILE_NAME = sorted(os.listdir(FOLDER_INPUT), key=FUNCTION_NATURAL_SORT_KEY)
    LIST_WORK_ITEM = []
    for EACH_INSTRUCTION in LIST_INSTRUCTION:
        SET_KEY = {EACH_INSTRUCTION, EACH_INSTRUCTION.lower()}
        if EACH_INSTRUCTION.startswith('AMO'):
            SET_KEY.add('AMO')
        elif '.' in EACH_INSTRUCTION and EACH_INSTRUCTION.index('.') >= 3:
            SET_KEY.add(EACH_INSTRUCTION.split('.', 1)[0])
        LIST_PATH = []
        for EACH_FILE_NAME in LIST_FILE_NAME:
            if not EACH_FILE_NAME.endswith('.txt'):
                continue
            FILE_PATH = os.path.join(FOLDER_INPUT, EACH_FILE_NAME)
            with open(FILE_PATH, encoding='utf-8') as FILE_HANDLE:
                if any(any(EACH_KEY in EACH_LINE for EACH_KEY in SET_KEY) for EACH_LINE in FILE_HANDLE):
                    LIST_PATH.append(FILE_PATH)
        LIST_PATH.sort(key=lambda EACH_PATH_VALUE: int(EACH_PATH_VALUE.split('SPECIFICATION_')[1].split('.txt')[0]))
        for LOCAL_NUMBER, EACH_FILE_PATH in enumerate(LIST_PATH, 1):
            LIST_WORK_ITEM.append({'instruction': EACH_INSTRUCTION, 'file_path': EACH_FILE_PATH, 'local_number': LOCAL_NUMBER})
    return LIST_WORK_ITEM

def FUNCTION_WORK_SHA256(LIST_WORK_ITEM):
    HASH_OBJECT = hashlib.sha256()
    with open(FILE_INPUT, 'rb') as FILE_HANDLE:
        for EACH_CHUNK in iter(lambda: FILE_HANDLE.read(1024 * 1024), b''):
            HASH_OBJECT.update(EACH_CHUNK)
    SET_HASHED_PATH = set()
    for EACH_WORK_ITEM in LIST_WORK_ITEM:
        EACH_FILE_PATH = EACH_WORK_ITEM['file_path']
        HASH_OBJECT.update((EACH_WORK_ITEM['instruction'] + '\0' + EACH_FILE_PATH + '\0' + str(EACH_WORK_ITEM['local_number']) + '\n').encode('utf-8'))
        if EACH_FILE_PATH in SET_HASHED_PATH:
            continue
        SET_HASHED_PATH.add(EACH_FILE_PATH)
        with open(EACH_FILE_PATH, 'rb') as FILE_HANDLE:
            for EACH_CHUNK in iter(lambda: FILE_HANDLE.read(1024 * 1024), b''):
                HASH_OBJECT.update(EACH_CHUNK)
    return HASH_OBJECT.hexdigest()

def FUNCTION_CHECKPOINT(NEXT_INDEX):
    global LIST_BUFFER_HISTORY, LIST_BUFFER_OUTPUT, LOOPS_SINCE_CHECKPOINT
    if LIST_BUFFER_HISTORY:
        STRING_HISTORY = SUFFIX.join(LIST_BUFFER_HISTORY)
        if os.path.getsize(FILE_PROMPT) > 0:
            STRING_HISTORY = SUFFIX + STRING_HISTORY
        FUNCTION_APPEND_AND_SYNC(FILE_PROMPT, STRING_HISTORY)
    if LIST_BUFFER_OUTPUT:
        STRING_OUTPUT = '\n'.join(LIST_BUFFER_OUTPUT)
        if os.path.getsize(FILE_OUTPUT) > 0:
            STRING_OUTPUT = '\n' + STRING_OUTPUT
        FUNCTION_APPEND_AND_SYNC(FILE_OUTPUT, STRING_OUTPUT)
    NEW_STATE = {
        'next_index': NEXT_INDEX,
        'next_loop': NEXT_INDEX + 1,
        'global_number': GLOBAL_NUMBER,
        'work_sha256': WORK_SHA256,
        'paths': {'file_output': FILE_OUTPUT, 'file_prompt': FILE_PROMPT},
        'file_sizes': {'file_output': os.path.getsize(FILE_OUTPUT), 'file_prompt': os.path.getsize(FILE_PROMPT)}
    }
    FUNCTION_ATOMIC_WRITE_JSON(FILE_CHECKPOINT, NEW_STATE)
    print(f'[CHECKPOINT] Saved through work item {NEXT_INDEX} of {len(LIST_WORK_ITEM)}.', flush=True)
    LIST_BUFFER_HISTORY = []
    LIST_BUFFER_OUTPUT = []
    LOOPS_SINCE_CHECKPOINT = 0
# FUNCTION - FINISH

# VARIABLE - BEGIN
DIVIDER_LINE = ('#' * WIDTH + '\n') * 3
DIVIDER_LINE = DIVIDER_LINE.rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
LIST_BUFFER_HISTORY = []
LIST_BUFFER_OUTPUT = []
LOOPS_SINCE_CHECKPOINT = 0
SET_COMPRESS = {'add', 'addi', 'addi16sp', 'addi4spn', 'addiw', 'addw', 'and', 'andi', 'beqz', 'bnez', 'ebreak', 'fld', 'fldsp', 'flw', 'flwsp', 'fsd', 'fsdsp', 'fsw', 'fswsp', 'j', 'jal', 'jalr', 'jr', 'lbu', 'ld', 'ldsp', 'lh', 'lhu', 'li', 'lui', 'lw', 'lwsp', 'mop.n', 'mul', 'mv', 'nop', 'not', 'ntl.all', 'ntl.p1', 'ntl.pall', 'ntl.s1', 'or', 'sb', 'sd', 'sdsp', 'sext.b', 'sext.h', 'sh', 'slli', 'srai', 'srli', 'sub', 'subw', 'sw', 'swsp', 'xor', 'zext.b', 'zext.h', 'zext.w'}
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_FILE_PATH in [FILE_OUTPUT, FILE_PROMPT, FILE_CHECKPOINT]:
    Path(EACH_FILE_PATH).parent.mkdir(parents=True, exist_ok=True)

if RESET_PROGRESS and Path(FILE_CHECKPOINT).exists():
    Path(FILE_CHECKPOINT).unlink()

LIST_WORK_ITEM = FUNCTION_BUILD_WORK_ITEMS()
WORK_SHA256 = FUNCTION_WORK_SHA256(LIST_WORK_ITEM)

if Path(FILE_CHECKPOINT).exists():
    with open(FILE_CHECKPOINT, encoding='utf-8') as FILE_HANDLE:
        STATE = json.load(FILE_HANDLE)
    if STATE.get('work_sha256') != WORK_SHA256:
        raise RuntimeError('Instruction/specification inputs changed after the checkpoint was created. Refusing to resume because that could duplicate or misalign output. Set RESET_PROGRESS = True only if you intentionally want a fresh run.')
    for EACH_PATH_KEY, EACH_FILE_PATH in [('file_output', FILE_OUTPUT), ('file_prompt', FILE_PROMPT)]:
        SAVED_PATH = STATE.get('paths', {}).get(EACH_PATH_KEY)
        if SAVED_PATH != EACH_FILE_PATH:
            raise RuntimeError(f'Checkpoint path mismatch for {EACH_PATH_KEY}: {SAVED_PATH!r} != {EACH_FILE_PATH!r}')
    FUNCTION_RESTORE_COMMITTED_SIZE(FILE_OUTPUT, STATE['file_sizes']['file_output'])
    FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PROMPT, STATE['file_sizes']['file_prompt'])
    START_INDEX = STATE['next_index']
    GLOBAL_NUMBER = STATE['global_number']
    print(f'[RESUME] Continuing from work item {START_INDEX + 1} of {len(LIST_WORK_ITEM)}.', flush=True)
else:
    for EACH_FILE_PATH in [FILE_OUTPUT, FILE_PROMPT]:
        with open(EACH_FILE_PATH, 'w', encoding='utf-8'):
            pass
    START_INDEX = 0
    GLOBAL_NUMBER = 1
    STATE = {
        'next_index': 0,
        'next_loop': 1,
        'global_number': GLOBAL_NUMBER,
        'work_sha256': WORK_SHA256,
        'paths': {'file_output': FILE_OUTPUT, 'file_prompt': FILE_PROMPT},
        'file_sizes': {'file_output': 0, 'file_prompt': 0}
    }
    FUNCTION_ATOMIC_WRITE_JSON(FILE_CHECKPOINT, STATE)

for EACH_INDEX_WORK in range(START_INDEX, len(LIST_WORK_ITEM)):
    EACH_WORK_ITEM = LIST_WORK_ITEM[EACH_INDEX_WORK]
    EACH_INSTRUCTION = EACH_WORK_ITEM['instruction']
    EACH_FILE_PATH = EACH_WORK_ITEM['file_path']
    LOCAL_NUMBER = EACH_WORK_ITEM['local_number']
    with open(EACH_FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        STRING_SPEC = FILE_HANDLE.read()
    STRING_EXCEPTION = ''
    if not EACH_INSTRUCTION.startswith('C.'):
        STRING_EXCEPTION = 'If the text contains any information about HINT instructions, output NO!\n\n'
    STRING_PROMPT = (
        f'For RISC-V, does this text below contain any information about {EACH_INSTRUCTION} instruction? '
        f'Output YES or NO only. '
        f'Output YES if the text explicitly mentions {EACH_INSTRUCTION}, or contains a table row, sentence, encoding statement, side-effect statement, read/write behavior, or pseudoinstruction expansion involving {EACH_INSTRUCTION}. '
        f'{STRING_EXCEPTION}'
        f'The text:\n\n{STRING_SPEC}'
    )
    STRING_RESPONSE = FUNCTION_ASK(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT, TEMPERATURE, CONTEXT)
    STRING_RESPONSE = FUNCTION_CLEAN_TEXT(STRING_RESPONSE)
    LIST_BUFFER_HISTORY.append(FUNCTION_WRAP_HISTORY(WIDTH, GLOBAL_NUMBER, LOCAL_NUMBER, EACH_INSTRUCTION, STRING_PROMPT, STRING_RESPONSE))
    if 'YES' in STRING_RESPONSE:
        LIST_BUFFER_OUTPUT.append(f'{EACH_INSTRUCTION} - {EACH_FILE_PATH}')
    GLOBAL_NUMBER += 1
    LOOPS_SINCE_CHECKPOINT += 1
    IS_LAST_WORK_ITEM = EACH_INDEX_WORK == len(LIST_WORK_ITEM) - 1
    if LOOPS_SINCE_CHECKPOINT >= CHECKPOINT_EVERY or IS_LAST_WORK_ITEM:
        FUNCTION_CHECKPOINT(EACH_INDEX_WORK + 1)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 3 - CREATE - 3_INSTRUCTION_BOOLEAN.txt - END

[CHECKPOINT] Saved through work item 10 of 32.
[CHECKPOINT] Saved through work item 20 of 32.
[CHECKPOINT] Saved through work item 30 of 32.
[CHECKPOINT] Saved through work item 32 of 32.
24.59 second(s)

====
DONE
====


In [6]:
# 4 - CREATE - 4_INSTRUCTION_RULE.txt - START

"""
Extract normative architectural behaviors from relevant specification sections, transform them into atomic and self-contained rules, and filter out non-actionable rules using LLMs.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from collections import defaultdict
from ollama import chat, Client, ResponseError
from pathlib import Path
import hashlib, json, os, re, subprocess
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '4_INSTRUCTION/1_OUTPUT/3_INSTRUCTION_BOOLEAN.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/4_INSTRUCTION_RULE.txt'
FILE_PROMPT_1 = '4_INSTRUCTION/2_HISTORY/2_HISTORY_INSTRUCTION_RULE_1.txt'
FILE_PROMPT_2 = '4_INSTRUCTION/2_HISTORY/3_HISTORY_INSTRUCTION_RULE_2.txt'
FILE_CHECKPOINT = '4_INSTRUCTION/3_WHICH_LOOP/WHICH_LOOP_RULE.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_CLEAN_TEXT(STRING_VALUE):
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+').replace("’", "'").replace("‘", "'"))

def FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR):
    if isinstance(ERROR, ConnectionError):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    STRING_ERROR = str(ERROR).lower()
    LIST_RETRY_KEYWORD = ['cuda', 'gpu', 'launch timed out', 'ggml_backend_cuda', 'cudastreamsynchronize', 'cannot access memory', 'runner']
    if STATUS_CODE is not None and STATUS_CODE >= 500:
        return True
    return any(EACH_KEYWORD in STRING_ERROR for EACH_KEYWORD in LIST_RETRY_KEYWORD)

def FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR):
    if isinstance(ERROR, ConnectionError):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    return STATUS_CODE in (408, 429) or (STATUS_CODE is not None and STATUS_CODE >= 500)

def FUNCTION_STOP_LOCAL_RUNNER(MODEL):
    print(f'[RECOVERY] Stopping Ollama runner for {MODEL!r}...', flush=True)
    try:
        COMPLETED_PROCESS = subprocess.run(['ollama', 'stop', MODEL], check=False, capture_output=True, text=True, timeout=RUNNER_STOP_TIMEOUT_SECONDS)
        if COMPLETED_PROCESS.returncode != 0:
            STRING_ERROR = COMPLETED_PROCESS.stderr.strip()
            if STRING_ERROR:
                print(f'[RECOVERY] ollama stop returned {COMPLETED_PROCESS.returncode}: {STRING_ERROR}', flush=True)
    except Exception as ERROR:
        print(f'[RECOVERY] Could not stop runner cleanly: {ERROR}', flush=True)

def FUNCTION_ASK(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT, TEMPERATURE, CONTEXT):
    NUMBER_ATTEMPT = 0
    while True:
        NUMBER_ATTEMPT += 1
        try:
            if IS_LOCAL:
                return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content.strip()
            return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content.strip()
        except (ResponseError, ConnectionError) as ERROR:
            IS_RETRYABLE = FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR) if IS_LOCAL else FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR)
            if not AUTO_RETRY or not IS_RETRYABLE or NUMBER_ATTEMPT >= MAX_RETRIES:
                print(f'[RECOVERY] Giving up after attempt {NUMBER_ATTEMPT}/{MAX_RETRIES}.', flush=True)
                raise
            STRING_MODE = 'LOCAL' if IS_LOCAL else 'ONLINE'
            print(f'[RECOVERY] {STRING_MODE} request failed on attempt {NUMBER_ATTEMPT}/{MAX_RETRIES}: {ERROR}', flush=True)
            if IS_LOCAL:
                FUNCTION_STOP_LOCAL_RUNNER(MODEL)
            print(f'[RECOVERY] Waiting {RETRY_WAIT_SECONDS} seconds, then retrying the SAME request automatically...', flush=True)
            time.sleep(RETRY_WAIT_SECONDS)

def FUNCTION_WRAP_HISTORY(WIDTH, GLOBAL_NUMBER, LOCAL_NUMBER, STRING_NAME, STRING_PROMPT, STRING_RESPONSE, STRING_SUFFIX):
    HALF_WIDTH = WIDTH // 2
    INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RESPONSE}
'''
    return INTRO_FIRST + '\n' + INTRO + STRING_PROMPT + ENDING + INTRO_END + STRING_SUFFIX

def FUNCTION_SHA256(FILE_PATH):
    HASH_OBJECT = hashlib.sha256()
    with open(FILE_PATH, 'rb') as FILE_HANDLE:
        for EACH_CHUNK in iter(lambda: FILE_HANDLE.read(1024 * 1024), b''):
            HASH_OBJECT.update(EACH_CHUNK)
    return HASH_OBJECT.hexdigest()

def FUNCTION_WORK_SHA256(LIST_INPUT_CONTENT):
    HASH_OBJECT = hashlib.sha256()
    HASH_OBJECT.update(FUNCTION_SHA256(FILE_INPUT).encode('ascii'))
    SET_HASHED_PATH = set()
    for EACH_STRING_LINE in LIST_INPUT_CONTENT:
        FILE_PATH = EACH_STRING_LINE.split(' - ', 1)[1].strip()
        if FILE_PATH in SET_HASHED_PATH:
            continue
        SET_HASHED_PATH.add(FILE_PATH)
        HASH_OBJECT.update(FILE_PATH.encode('utf-8') + b'\0')
        HASH_OBJECT.update(FUNCTION_SHA256(FILE_PATH).encode('ascii'))
    return HASH_OBJECT.hexdigest()

def FUNCTION_ATOMIC_WRITE(FILE_PATH, STRING_VALUE):
    FILE_PATH = Path(FILE_PATH)
    FILE_TEMP = FILE_PATH.with_suffix(FILE_PATH.suffix + '.tmp')
    with open(FILE_TEMP, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_VALUE)
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())
    os.replace(FILE_TEMP, FILE_PATH)

def FUNCTION_APPEND_HISTORY(FILE_PATH, STRING_VALUE):
    if not STRING_VALUE:
        return
    STRING_VALUE = STRING_VALUE.removesuffix(SUFFIX)
    if os.path.getsize(FILE_PATH) > 0:
        STRING_VALUE = SUFFIX + STRING_VALUE
    with open(FILE_PATH, 'a', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_VALUE)
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())

def FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PATH, COMMITTED_SIZE):
    FILE_PATH = Path(FILE_PATH)
    if not FILE_PATH.exists():
        if COMMITTED_SIZE != 0:
            raise RuntimeError(f'Cannot resume safely: {FILE_PATH} is missing.')
        FILE_PATH.touch()
        return
    CURRENT_SIZE = FILE_PATH.stat().st_size
    if CURRENT_SIZE < COMMITTED_SIZE:
        raise RuntimeError(f'Cannot resume safely: {FILE_PATH} is smaller than its committed size.')
    if CURRENT_SIZE > COMMITTED_SIZE:
        with open(FILE_PATH, 'r+b') as FILE_HANDLE:
            FILE_HANDLE.truncate(COMMITTED_SIZE)
            FILE_HANDLE.flush()
            os.fsync(FILE_HANDLE.fileno())

def FUNCTION_WRITE_OUTPUT():
    MAP_INSTRUCTION_RULE = defaultdict(list)
    LIST_OUTPUT_FINAL = []
    LIST_OUTPUT_SUPER_FINAL = []
    for EACH_STRING_CONTENT in LIST_OUTPUT_CONTENT:
        LIST_LINE = EACH_STRING_CONTENT.strip().splitlines()
        STRING_NAME = LIST_LINE[0].split(':', 1)[1].strip()
        MAP_INSTRUCTION_RULE[STRING_NAME].extend(LIST_LINE[1:])
    for EACH_STRING_NAME, EACH_LIST_RULE in MAP_INSTRUCTION_RULE.items():
        LIST_LINE_NEW = [f'INSTRUCTION: {EACH_STRING_NAME}']
        for EACH_INDEX_RULE, EACH_RULE_LINE in enumerate(EACH_LIST_RULE, 1):
            STRING_RULE_TEXT = EACH_RULE_LINE.split(':', 1)[1].strip()
            LIST_LINE_NEW.append(f'RULE {EACH_INDEX_RULE}: {STRING_RULE_TEXT}')
        LIST_OUTPUT_FINAL.append('\n'.join(LIST_LINE_NEW))
    for EACH_STRING_EACH in LIST_OUTPUT_FINAL:
        LIST_LINE = EACH_STRING_EACH.splitlines()
        STRING_INSTRUCTION_LINE = LIST_LINE[0].strip()
        LIST_RULE = [EACH_LN.strip() for EACH_LN in LIST_LINE[1:] if EACH_LN.strip() and EACH_LN.strip() != 'RULE:']
        LIST_RULE = [re.sub(r'(RULE\s+\d+:)(\S)', r'\1 \2', EACH_LN) for EACH_LN in LIST_RULE]
        LIST_RENUMBERED = []
        for EACH_INDEX_RULE, EACH_RULE_LINE in enumerate(LIST_RULE, 1):
            LIST_RENUMBERED.append(re.sub(r'RULE\s+\d+:', f'RULE {EACH_INDEX_RULE}:', EACH_RULE_LINE))
        LIST_OUTPUT_SUPER_FINAL.append(f'{STRING_INSTRUCTION_LINE}\n' + '\n'.join(LIST_RENUMBERED))
    FUNCTION_ATOMIC_WRITE(FILE_OUTPUT, '\n\n'.join(LIST_OUTPUT_SUPER_FINAL))

def FUNCTION_CHECKPOINT(NEXT_INDEX):
    global STRING_PROMPT_HISTORY_1, STRING_PROMPT_HISTORY_2
    FUNCTION_APPEND_HISTORY(FILE_PROMPT_1, STRING_PROMPT_HISTORY_1)
    FUNCTION_APPEND_HISTORY(FILE_PROMPT_2, STRING_PROMPT_HISTORY_2)
    NEW_STATE = {
        'next_index': NEXT_INDEX,
        'next_loop': NEXT_INDEX + 1,
        'global_number_1': GLOBAL_NUMBER_1,
        'local_number_1': LOCAL_NUMBER_1,
        'global_number_2': GLOBAL_NUMBER_2,
        'local_number_2': LOCAL_NUMBER_2,
        'instruction_initial': INSTRUCTION_INITIAL,
        'work_sha256': WORK_SHA256,
        'list_output_content': LIST_OUTPUT_CONTENT,
        'paths': {
            'file_output': FILE_OUTPUT,
            'file_prompt_1': FILE_PROMPT_1,
            'file_prompt_2': FILE_PROMPT_2
        },
        'file_sizes': {
            'file_prompt_1': os.path.getsize(FILE_PROMPT_1),
            'file_prompt_2': os.path.getsize(FILE_PROMPT_2)
        }
    }
    FUNCTION_ATOMIC_WRITE(FILE_CHECKPOINT, json.dumps(NEW_STATE, indent=2, sort_keys=True) + '\n')
    STRING_PROMPT_HISTORY_1 = ''
    STRING_PROMPT_HISTORY_2 = ''
    FUNCTION_WRITE_OUTPUT()
    print(f'[CHECKPOINT] Saved through input loop {NEXT_INDEX} of {len(LIST_INPUT_CONTENT)}.', flush=True)
# FUNCTION - FINISH

# VARIABLE - BEGIN
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
LIST_OUTPUT_CONTENT = []
STRING_PROMPT_HISTORY_1 = ''
STRING_PROMPT_HISTORY_2 = ''
GLOBAL_NUMBER_1 = 1
LOCAL_NUMBER_1 = 1
GLOBAL_NUMBER_2 = 1
LOCAL_NUMBER_2 = 1
INSTRUCTION_INITIAL = None
# VARIABLE - FINISH

# CODE - BEGIN
for EACH_FILE_PATH in [FILE_OUTPUT, FILE_PROMPT_1, FILE_PROMPT_2, FILE_CHECKPOINT]:
    Path(EACH_FILE_PATH).parent.mkdir(parents=True, exist_ok=True)

with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    LIST_INPUT_CONTENT = FILE_HANDLE.readlines()

WORK_SHA256 = FUNCTION_WORK_SHA256(LIST_INPUT_CONTENT)

if RESET_PROGRESS and Path(FILE_CHECKPOINT).exists():
    Path(FILE_CHECKPOINT).unlink()

if Path(FILE_CHECKPOINT).exists():
    with open(FILE_CHECKPOINT, encoding='utf-8') as FILE_HANDLE:
        STATE = json.load(FILE_HANDLE)
    if STATE.get('work_sha256') != WORK_SHA256:
        raise RuntimeError('Instruction/specification inputs changed. Set RESET_PROGRESS = True only if you want a fresh run.')
    for EACH_PATH_KEY, EACH_FILE_PATH in [('file_output', FILE_OUTPUT), ('file_prompt_1', FILE_PROMPT_1), ('file_prompt_2', FILE_PROMPT_2)]:
        if STATE.get('paths', {}).get(EACH_PATH_KEY) != EACH_FILE_PATH:
            raise RuntimeError(f'Checkpoint path mismatch for {EACH_PATH_KEY}.')
    FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PROMPT_1, STATE['file_sizes']['file_prompt_1'])
    FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PROMPT_2, STATE['file_sizes']['file_prompt_2'])
    LIST_OUTPUT_CONTENT = STATE['list_output_content']
    START_INDEX = STATE['next_index']
    GLOBAL_NUMBER_1 = STATE['global_number_1']
    LOCAL_NUMBER_1 = STATE['local_number_1']
    GLOBAL_NUMBER_2 = STATE['global_number_2']
    LOCAL_NUMBER_2 = STATE['local_number_2']
    INSTRUCTION_INITIAL = STATE['instruction_initial']
    FUNCTION_WRITE_OUTPUT()
    print(f'[RESUME] Continuing from input loop {START_INDEX + 1} of {len(LIST_INPUT_CONTENT)}.', flush=True)
else:
    for EACH_FILE_PATH in [FILE_PROMPT_1, FILE_PROMPT_2]:
        FUNCTION_ATOMIC_WRITE(EACH_FILE_PATH, '')
    START_INDEX = 0
    FUNCTION_CHECKPOINT(0)

LOOPS_SINCE_CHECKPOINT = 0

for EACH_INDEX_LINE in range(START_INDEX, len(LIST_INPUT_CONTENT)):
    EACH_STRING_LINE = LIST_INPUT_CONTENT[EACH_INDEX_LINE]
    INSTRUCTION_NAME = EACH_STRING_LINE.split(' - ', 1)[0]
    FILE_PATH = EACH_STRING_LINE.split(' - ', 1)[1].strip()
    if EACH_INDEX_LINE == 0:
        INSTRUCTION_INITIAL = INSTRUCTION_NAME
    elif INSTRUCTION_NAME != INSTRUCTION_INITIAL:
        INSTRUCTION_INITIAL = INSTRUCTION_NAME
        LOCAL_NUMBER_1 = 1
        LOCAL_NUMBER_2 = 1
    with open(FILE_PATH, encoding='utf-8') as FILE_HANDLE:
        STRING_SPECIFICATION = FILE_HANDLE.read()

    # PROMPT 1 - BEGIN
    STRING_PROMPT_1 = f'''For RISC-V, please extract all possible information about {INSTRUCTION_NAME} instruction from the text below. The information must be only about {INSTRUCTION_NAME} instruction, not other things. Extract ONLY explicit, testable, normative description about architectural behavior.

Please output in paragraphs. If you want, you can include bullet or numbered points in your output. You can also include tables. However, do not give me any introduction in your output. You just need to output the information that you extract.

If your output includes some sentences that refer to some tables or figures, include those tables or figures too.

Remember, output must be ASCII-only text.

If it happens to be there is no information about {INSTRUCTION_NAME} instruction from the text below, output NONE

Preserve the original normative strength exactly: use MUST/MUST NOT only when the source uses MUST/MUST NOT, and use SHOULD/SHOULD NOT only when the source uses SHOULD/SHOULD NOT; never upgrade or downgrade normative keywords.

Your description must be very clear and unambiguous. If you want, you can output in bullet points also.

Use the exact terminology and wording used in the text below; do not substitute synonyms or paraphrase architectural terms.

If other instructions are explicitly stated to require the same permissions or raise the same exceptions as {INSTRUCTION_NAME}, you MUST include those shared behaviors as properties of {INSTRUCTION_NAME}.

Statements describing what IEEE 754-2008 permits or requires are about that standard, not about {INSTRUCTION_NAME}. Extract only the RISC-V behavior stated in the normative text, even when it differs from IEEE 754-2008.

The text below:

{STRING_SPECIFICATION}'''
    STRING_RESPONSE_1 = FUNCTION_ASK(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT_1, TEMPERATURE, CONTEXT)
    if STRING_RESPONSE_1.strip() == 'NONE':
        LOOPS_SINCE_CHECKPOINT += 1
        if LOOPS_SINCE_CHECKPOINT >= CHECKPOINT_EVERY or EACH_INDEX_LINE == len(LIST_INPUT_CONTENT) - 1:
            FUNCTION_CHECKPOINT(EACH_INDEX_LINE + 1)
            LOOPS_SINCE_CHECKPOINT = 0
        continue
    STRING_RESPONSE_1 = FUNCTION_CLEAN_TEXT(STRING_RESPONSE_1)
    STRING_PROMPT_HISTORY_1 += FUNCTION_WRAP_HISTORY(WIDTH, GLOBAL_NUMBER_1, LOCAL_NUMBER_1, INSTRUCTION_NAME, STRING_PROMPT_1, STRING_RESPONSE_1, SUFFIX)
    # PROMPT 1 - FINISH

    # PROMPT 2 - BEGIN
    STRING_PROMPT_2 = f'''For RISC-V, for {INSTRUCTION_NAME} instruction, rewrite the following description as a list of rules. Each rule must explain about {INSTRUCTION_NAME} instruction. Essentially, for each rule, it is about {INSTRUCTION_NAME} instruction, not other things.

Do not lose any information about {INSTRUCTION_NAME} instruction from the description. Read the text or the description carefully. Understand the context and the sentences. Do not lose any important information.

For each rule, do not use any internal references like "According to Table 7," or "see Figure 3." Each rule must be one sentence only. Each rule must stand fully on its own without cross-references. Do not quote or refer to other rules. Essentially, each rule is one idea. Avoid a rule like this: "The instruction being described is {INSTRUCTION_NAME}."

Do NOT refer to other rules by number, name, or description. Phrases such as "described in RULE N", "in the previous rule", "the rule above", "as stated earlier", or "the same condition as RULE N" are FORBIDDEN. If two rules share context or conditions, restate the shared context fully in each rule rather than referencing the other rule. Each rule must be fully self-contained and understandable in isolation, even if read out of order or extracted individually.

Each rule should become a single testable statement.

Each rule should be a separate atomic rule that expresses one architectural behavior or one event -> outcome relationship.

If a sentence contains multiple independent events or clearly different behaviors, split them into separate rules so that each rule expresses one case.

However, if a sentence describes a single event that can legitimately produce multiple alternative outcomes (e.g., "A may cause X or Y"), do NOT split the rule. In such cases, keep the alternatives in a single rule, because the specification is describing implementation-dependent outcomes where either result is acceptable.

Each rule needs to be clear and detailed. Extract ONLY explicit, testable, normative rules about architectural behavior. A rule MUST describe behavioral semantics, not naming, encoding, or syntax.

Remember, rules must describe architectural cause -> effect semantics (event -> architectural outcome).

Remember, if a sentence contains multiple distinct behaviors or enumerated cases, split them into multiple rules so each rule expresses only one case.

Exclude purely structural or descriptive properties (such as register width, naming, or existence) unless they imply observable architectural behavior.

Preserve the original normative strength as much as possible:
- Use MUST/MUST NOT only when the source clearly expresses a strict architectural requirement.
- For reserved encodings or unsupported cases, prefer describing them as "reserved" or "illegal" behavior instead of forcing "MUST NOT be implemented", unless the source explicitly mandates it.
- Do not unnecessarily strengthen the wording beyond what is needed for architectural correctness.

Special handling for specification terms:
- If the specification uses the term "reserved", preserve it exactly as "reserved".
- Do NOT reinterpret "reserved" as "illegal", "must trap", or "MUST NOT be implemented".
- A "reserved" encoding means the behavior is not defined by the specification, and the rule must not assume any specific outcome (such as an exception) unless explicitly stated.
- Only use "illegal instruction" or exception behavior if the specification explicitly states it.

OUTPUT FORMAT:
+ Each rule must begin with the word "RULE", followed by its number (e.g., "RULE 1: ", "RULE 2: ", "RULE 3: ", ...).
+ Do not give me any introduction.
+ Each rule must contain the word {INSTRUCTION_NAME}.

Remember, output must be ASCII-only text.

The following description:

{STRING_RESPONSE_1}'''
    STRING_RESPONSE_2 = FUNCTION_CLEAN_TEXT(FUNCTION_ASK(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT_2, TEMPERATURE, CONTEXT))
    STRING_PROMPT_HISTORY_2 += FUNCTION_WRAP_HISTORY(WIDTH, GLOBAL_NUMBER_2, LOCAL_NUMBER_2, INSTRUCTION_NAME, STRING_PROMPT_2, STRING_RESPONSE_2, SUFFIX)
    # PROMPT 2 - FINISH

    # PROMPT 3 - START
    STRING_PROMPT_3 = f'''Given the following list of rules about a RISC-V CSR, remove any rule that is not actionable. A rule is actionable if it describes a concrete observable behavior that can be verified by executing instructions on a RISC-V implementation, such as: the value read back from the CSR, whether an exception is raised or not raised, which privilege modes can access the CSR, how the CSR value changes after specific operations, or whether specific field values are reserved, invalid, or constrained.

Keep rules about reserved, invalid, or constrained CSR field values when they identify specific values or value ranges for a concrete field, because these can be tested by writing those values and observing readback behavior, trapping behavior, or other architectural effects.

Remove any rule that is purely identifying, such as CSR address encoding, count of CSRs in an extension, or abstract architectural purpose.

Do not rewrite, rephrase, or add to any rule. Output only the surviving rules exactly as they appeared in the input, renumbered starting from 1.

Each output rule must begin with "RULE" followed by its number. Output must be ASCII-only text. Do not give any introduction.

{STRING_RESPONSE_2}'''
    STRING_RESPONSE_3 = FUNCTION_CLEAN_TEXT(FUNCTION_ASK(IS_LOCAL, MODEL, CLIENT, STRING_PROMPT_3, TEMPERATURE, CONTEXT))
    # PROMPT 3 - FINISH

    # FINALIZING - BEGIN
    STRING_RESPONSE_FINISH = STRING_RESPONSE_3
    LIST_LINE_RULE = [EACH_LN.strip() for EACH_LN in STRING_RESPONSE_FINISH.splitlines() if EACH_LN.strip()]
    LIST_RULE = []
    for EACH_LINE_RULE in LIST_LINE_RULE:
        if re.match(r'^RULE\s+\d+:', EACH_LINE_RULE, re.IGNORECASE):
            EACH_LINE_RULE = re.sub(r'(RULE\s+\d+:)(\S)', r'\1 \2', EACH_LINE_RULE)
            LIST_RULE.append(EACH_LINE_RULE)
    LIST_LINE_FIXED = []
    for EACH_LINE_RULE in LIST_RULE:
        MATCH_RULE = re.match(r'^\s*RULE\s+(\d+):\s*(.*)', EACH_LINE_RULE)
        if not MATCH_RULE:
            LIST_LINE_FIXED.append(EACH_LINE_RULE)
            continue
        NUMBER_RULE = MATCH_RULE.group(1)
        STRING_REST = MATCH_RULE.group(2)
        STRING_REST = re.sub(rf'^RULE\s+{NUMBER_RULE}:\s*', '', STRING_REST)
        LIST_LINE_FIXED.append(f'RULE {NUMBER_RULE}: {STRING_REST}')
    STRING_CLEANED = '\n'.join(LIST_LINE_FIXED).replace('\u2011', '-')
    LIST_OUTPUT_CONTENT.append(f'INSTRUCTION: {INSTRUCTION_NAME}\n{STRING_CLEANED}')
    # FINALIZING - FINISH

    # CLEANING UP PROMPT - BEGIN
    GLOBAL_NUMBER_1 += 1
    LOCAL_NUMBER_1 += 1
    GLOBAL_NUMBER_2 += 1
    LOCAL_NUMBER_2 += 1
    # CLEANING UP PROMPT - FINISH

    # CHECKPOINT - BEGIN
    LOOPS_SINCE_CHECKPOINT += 1
    if LOOPS_SINCE_CHECKPOINT >= CHECKPOINT_EVERY or EACH_INDEX_LINE == len(LIST_INPUT_CONTENT) - 1:
        FUNCTION_CHECKPOINT(EACH_INDEX_LINE + 1)
        LOOPS_SINCE_CHECKPOINT = 0
    # CHECKPOINT - FINISH
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 4 - CREATE - 4_INSTRUCTION_RULE.txt - END

[CHECKPOINT] Saved through input loop 0 of 24.
[CHECKPOINT] Saved through input loop 10 of 24.
[CHECKPOINT] Saved through input loop 20 of 24.
[CHECKPOINT] Saved through input loop 24 of 24.
13 minute(s) 1.43 second(s)

====
DONE
====


In [7]:
# 5 - CREATE - 5_INSTRUCTION_FINAL.txt - START

"""
Merge instruction formats with extracted rules, filter rules based on the target RV64 configuration and test harness capabilities, and refine them using LLMs for automatic assembly test case generation.
"""

# TIME START - BEGIN
import time, re, os
TIME_START = time.time()
# TIME START - FINISH

# PATH - BEGIN
FILE_INPUT_NAME = '4_INSTRUCTION/1_OUTPUT/2_INSTRUCTION_SOME.txt'
FILE_INPUT_RULE = '4_INSTRUCTION/1_OUTPUT/4_INSTRUCTION_RULE.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/5_INSTRUCTION_FINAL.txt'
# PATH - FINISH

# CODE - BEGIN
with open(FILE_INPUT_NAME, encoding='utf-8') as FILE_HANDLE:
    LIST_NAME = FILE_HANDLE.readlines()

with open(FILE_INPUT_RULE, encoding='utf-8') as FILE_HANDLE:
    LIST_OUTPUT_LINE = FILE_HANDLE.readlines()

for EACH_INDEX_LINE, EACH_STRING_LINE in enumerate(LIST_NAME):
    if EACH_STRING_LINE.startswith('INSTRUCTION: '):
        try:
            INDEX_MATCH = LIST_OUTPUT_LINE.index(EACH_STRING_LINE)
        except ValueError:
            continue
        LIST_OUTPUT_LINE.insert(INDEX_MATCH + 1, LIST_NAME[EACH_INDEX_LINE + 1].rstrip() + '\n')

from ollama import chat, Client

LIST_NOT_IMPLEMENTED = ['sireg', 'sireg2', 'sireg3', 'sireg4', 'sireg5', 'sireg6', 'vsireg', 'vsireg2', 'vsireg3', 'vsireg4', 'vsireg5', 'vsireg6', 'jvt', 'mnstatus', 'mncause', 'tcontrol', 'mireg', 'mireg2', 'mireg3', 'mireg4', 'mireg5', 'mireg6', 'mtopei', *[f'hpmcounter{i}' for i in range(19, 32)], *[f'hpmcounter{i}h' for i in range(3, 32)], *[f'mhpmcounter{i}' for i in range(19, 32)], *[f'mhpmcounter{i}h' for i in range(3, 32)], 'pmpcfg1', 'pmpcfg3', 'pmpcfg5', 'pmpcfg7', 'pmpcfg9', 'pmpcfg11', 'pmpcfg13', 'pmpcfg15', 'mctrctl', 'mcyclecfg', 'mcyclecfgh', 'mcycleh', 'medelegh', 'menvcfgh', *[f'mhpmevent{i}h' for i in range(3, 32)], 'midelegh', 'mieh', 'minstretcfg', 'minstretcfgh', 'minstreth', 'mintstatus', 'miph', 'mnepc', 'mnscratch', 'mnxti', 'mscratchcsw', 'mscratchcswl', 'mseccfgh', 'mstatush', 'mtvt', 'mvienh', 'mviph', 'hcontext', 'hedelegh', 'henvcfg', 'hidelegh', 'htimedeltah', 'hvienh', 'hviph', 'hviprio1h', 'hviprio2h', 'mscontext', 'scontext', 'sctrctl', 'sctrdepth', 'sctrstatus', 'sieh', 'sintstatus', 'siph', 'snxti', 'srmcfg', 'sscratchcsw', 'sscratchcswl', 'stimecmph', 'stopei', 'stvt', 'cycleh', 'instreth', 'timeh', 'uintstatus', 'unxti', 'uscratchcsw', 'uscratchcswl', 'utvt', 'vsieh', 'vsiph', 'vstimecmph', 'vstopei', 'scountovf', 'stimecmp', 'vstimecmp', 'hstateen0h', 'hstateen1h', 'hstateen2h', 'hstateen3h', 'mstateen0h', 'mstateen1h', 'mstateen2h', 'mstateen3h', 'aes32dsi', 'aes32dsmi', 'aes32esi', 'aes32esmi']
LIST_NOT_IMPLEMENTED += ['tdata3', 'tselect', 'hgeie', 'hgeip']
STRING_NOT_IMPLEMENTED = ', '.join(LIST_NOT_IMPLEMENTED)

LIST_FINAL_LINE = []
LIST_RULE_CURRENT = []
RULE_COUNTER = 1
CURRENT_INSTRUCTION = ''
CURRENT_FORMAT = ''

for EACH_LINE in LIST_OUTPUT_LINE + ['']:
    STRING_LINE = EACH_LINE.strip()

    IS_NEW_BLOCK = STRING_LINE.startswith('INSTRUCTION:')
    IS_FORMAT = STRING_LINE.startswith('FORMAT:')
    IS_NEW_RULE = STRING_LINE.startswith('RULE ')

    if (IS_NEW_BLOCK or IS_FORMAT or IS_NEW_RULE or not STRING_LINE) and LIST_RULE_CURRENT:
        STRING_RULE_FULL = ' '.join(EACH_RULE_LINE.strip() for EACH_RULE_LINE in LIST_RULE_CURRENT)
        EACH_RULE = STRING_RULE_FULL.split(':', 1)[1].strip()

        STRING_PROMPT = f'''You are given one RISC-V architectural instruction rule. Output YES or NO only. No additional words.

Current instruction:
{CURRENT_INSTRUCTION}

Current instruction format:
{CURRENT_FORMAT}

Kernel info:
XLEN = 64
MXLEN = 64
SXLEN = 64
UXLEN = 64
HSXLEN = 64
RV64
M-mode, S-mode, U-mode, and HS-mode are implemented and supported.
U-mode is implemented and supported.
S-mode is implemented and supported.
HS-mode is implemented and supported.
VS-mode and VU-mode are implemented and supported.
Most standard RISC-V extensions and registers are implemented.

Implemented extensions include:
rv64imafdcvh, zicsr, zicntr, zihpm, smstateen, smaia, sscofpmf, sstc, zicfiss, zicbom, zicboz, zca, zimop, zcmop, zbb, zbs, zkne, zbkb, zabha, zacas, zawrs, zkr, smepmp, zcb, zicond, zba, zknd, zbc, zbkc, zfh, zfbfmin, zfhmin, zfa, zifencei, zvfbfmin, zbkx, zvksed, zvksh, zvknha, zvknhb, zvkg, zvfbfwma, zvbc, zvbb, zvkned, zksed, zksh, zknh, zvkb, zicbop, zicfilp, svinval, zve32f.

These registers/instructions are not implemented:
{STRING_NOT_IMPLEMENTED}

The test harness can observe integer registers, floating-point registers, vector registers, CSRs, fflags/frm/fcsr, mcause/scause, privilege-mode trap behavior, and custom read-only/write-only CSR side effects.

Default policy:
- Prefer YES if the rule describes architectural behavior of the current instruction and the behavior can plausibly be checked through registers, CSRs, fflags, trap behavior, privilege behavior, or read/write side effects.
- Output NO only when the rule clearly cannot produce a simple observable architectural result or clearly belongs to a different instruction.

Instruction matching policy:
- Output YES only if the rule applies to the current instruction.
- Output NO if the rule is only about a different instruction mnemonic than the current instruction.
- For example, if the current instruction is CSRRW, output NO for a rule that only describes CSRRWI.
- For example, if the current instruction is CSRRWI, output NO for a rule that only describes CSRRW.
- If the rule mentions the current instruction and also mentions related instructions for comparison or shared behavior, evaluate the rule normally.
- If the rule describes a shared architectural behavior of an instruction family and the current instruction is clearly part of that family, evaluate the rule normally.
- Do not reject a rule only because it mentions a related instruction as context.
- Reject only when the behavioral requirement is clearly assigned to another instruction and not to the current instruction.

Output YES for rules about:
- integer/floating-point/vector register results
- floating-point exception flags or fcsr/fflags behavior
- instruction exceptions or trap/no-trap behavior
- CSR instruction semantics such as CSR read, CSR write, read-modify-write, rd=x0 suppressing reads, rs1/uimm behavior, and CSR side effects
- privilege-mode access behavior
- WARL/WLRL readback behavior
- instruction result computed from concrete operands

For CSR instruction rules, prefer YES when the rule applies to the current CSR instruction and describes:
- old CSR value written to rd
- rs1 value written to CSR for register-form CSR instructions
- uimm value written to CSR for immediate-form CSR instructions
- rs1=x0 causing zero to be written to the CSR for register-form CSR instructions
- uimm=0 causing zero to be written to the CSR for immediate-form CSR instructions
- rd=x0 suppressing CSR read
- read side effects or write side effects
- read-modify-write ordering
- explicit CSR writes overriding instruction side effects

For CSR instruction rules:
- Output NO if the rule is only about the immediate form of a CSR instruction and the current instruction is the register form.
- Output NO if the rule is only about the register form of a CSR instruction and the current instruction is the immediate form.
- Example: for current instruction CSRRW, a rule only saying "CSRRWI writes an XLEN-bit zero-extended 5-bit unsigned immediate" should be NO.
- Example: for current instruction CSRRWI, a rule only saying "CSRRW writes the initial value in rs1 to the CSR" should be NO.
- If the rule explicitly applies to both forms or describes generic CSR side-effect behavior shared by the forms, evaluate it normally.
- Output NO if the rule says that some other CSR field, mode change, or system state change takes effect without requiring the current instruction.
- Example: for current instruction SFENCE.VMA, "Changes to satp.ASID take effect immediately without requiring SFENCE.VMA" should be NO because it is about satp.ASID behavior, not SFENCE.VMA execution behavior.
- Example: for current instruction SFENCE.VMA, "Changes to satp.MODE take effect immediately without requiring SFENCE.VMA" should be NO.
- Example: for current instruction SFENCE.VMA, "Changes to sstatus SUM and MXR take effect immediately without requiring SFENCE.VMA" should be NO.

For floating-point instruction rules, prefer YES when the rule applies to the current floating-point instruction and describes:
- exception flags
- invalid-operation, divide-by-zero, overflow, underflow, or inexact behavior
- result value or classification behavior
- behavior matching another instruction in the same arithmetic family

For floating-point family rules:
- If the current instruction is a fused multiply-add instruction, output YES for rules that describe shared fused multiply-add exception behavior.
- Do not reject a rule only because it compares the current instruction with related instructions in the same arithmetic family.

Output NO if the rule only says that an instruction reads a source register or consumes an input operand, and the rule does not specify any resulting register value, CSR value, memory value, flag value, trap behavior, or side effect.
Do not apply this rejection to CSR instructions.

Output NO if the rule requires an advanced page-table/TLB/cache test rather than a simple architectural state test. In particular, output NO for rules mainly about:
- TLB/cache invalidation
- address-translation cache entries
- page-table walk ordering
- memory-management synchronization
- implicit reads of page tables
- whether an instruction applies to satp/vsatp/hgatp-controlled memory-management structures
- ASID/VMID translation-cache matching
- global mapping preservation or invalidation
- PMP/page-table synchronization
- stale translations
- changes becoming visible only through page-table translation behavior

Applicability rules:
- If the rule references only not-implemented registers/instructions, output NO unless the rule is explicitly about illegal access to an unimplemented instruction.
- If the rule says "may", "might", "can", "implementation-defined", or describes optional behavior, output NO unless it also states a required observable result.
- Do not reject a rule merely because it contains "may" or "might" inside a subordinate phrase, if the main rule states a required observable behavior.
- Example: "CSRRW does not read the CSR and prevents side effects that might occur on a CSR read" should be YES because the required behavior is that CSRRW does not read the CSR.
- If the rule is only a descriptive statement without any architectural effect, output NO.

For fence and memory-management instructions:
- Output YES only if the rule is directly about executing the current fence instruction and observing required exception behavior, illegal-instruction behavior, or a required no-trap condition under a specific architectural setup.
- Output NO if the rule says the current fence instruction is not required.
- Output NO if the rule says another CSR/state change takes effect immediately without requiring the current fence instruction.
- Output NO if the rule says "may" or "might" raise an exception.
- Output NO for ordering, synchronization, TLB/cache/page-table, VMID, global-mapping, source-register-read, or translation-cache effects.

Output NO if the rule is mainly about:
- performance, timing, microarchitectural state, or implementation-internal behavior
- reserved behavior without a concrete required observable result
- behavior that would require constructing page tables or testing actual address translation correctness

The rule:
{EACH_RULE}'''

        if IS_LOCAL:
            RESPONSE = chat(
                model=MODEL,
                messages=[{'role': 'user', 'content': STRING_PROMPT}],
                options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT},
                keep_alive='30m'
            ).message.content
        else:
            RESPONSE = CLIENT.chat(
                model=MODEL,
                messages=[{'role': 'user', 'content': STRING_PROMPT}],
                options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT},
                keep_alive='30m'
            ).message.content

        if RESPONSE.strip().upper().startswith('YES'):
            LIST_FINAL_LINE.append(f'RULE {RULE_COUNTER}: {EACH_RULE}\n')
            RULE_COUNTER += 1

        LIST_RULE_CURRENT = []

    if IS_NEW_BLOCK:
        CURRENT_INSTRUCTION = STRING_LINE.split(':', 1)[1].strip()
        CURRENT_FORMAT = ''
        RULE_COUNTER = 1
        if LIST_FINAL_LINE and LIST_FINAL_LINE[-1].strip():
            LIST_FINAL_LINE.append('\n')
        LIST_FINAL_LINE.append(EACH_LINE)
        continue

    if IS_FORMAT:
        CURRENT_FORMAT = STRING_LINE.split(':', 1)[1].strip()
        LIST_FINAL_LINE.append(EACH_LINE)
        continue

    if IS_NEW_RULE:
        LIST_RULE_CURRENT = [EACH_LINE]
        continue

    if LIST_RULE_CURRENT and STRING_LINE:
        LIST_RULE_CURRENT.append(EACH_LINE)
        continue

    if STRING_LINE:
        LIST_FINAL_LINE.append(EACH_LINE)

STRING_OUTPUT = ''.join(LIST_FINAL_LINE).strip() + '\n'

def FUNCTION_CALL_LLM(STRING_PROMPT):
    if IS_LOCAL:
        RESPONSE = chat(
            model=MODEL,
            messages=[{'role': 'user', 'content': STRING_PROMPT}],
            options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT},
            keep_alive='30m'
        ).message.content
    else:
        RESPONSE = CLIENT.chat(
            model=MODEL,
            messages=[{'role': 'user', 'content': STRING_PROMPT}],
            options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT},
            keep_alive='30m'
        ).message.content

    return RESPONSE.strip()

def FUNCTION_CHECK_RULE_TESTABLE(STRING_RULE):
    STRING_PROMPT = f'''You are given one RISC-V architectural rule.

Determine whether the rule is already clear, directly testable, and actionable for automatic assembly test generation.

Output only:
YES
or
NO

Output YES only if:
- the rule contains a clear architectural condition or setup
- the rule contains a clear observable architectural result
- the observable result can be tested using assembly
- the rule is specific enough to generate a concrete test case
- the rule describes behavior observable through:
  - registers
  - CSRs
  - trap behavior
  - mcause/scause
  - privilege behavior
  - instruction result
  - readback behavior
  - architectural exceptions

Output NO if:
- the rule is vague or descriptive
- the rule only describes metadata
- the rule states a constraint without an observable consequence
- the rule lacks a concrete expected result
- the rule is mainly about:
  - implementation details
  - microarchitectural behavior
  - performance
  - timing
  - cache/TLB internals
  - synchronization internals
  - implicit hardware behavior
- the rule is too abstract to generate a concrete assembly test

The rule:
{STRING_RULE}'''

    STRING_RESPONSE = FUNCTION_CALL_LLM(STRING_PROMPT)
    return STRING_RESPONSE.strip().upper().startswith('YES')

def FUNCTION_REWRITE_RULE_TESTABLE(STRING_RULE):
    STRING_PROMPT = f'''You are given one old RISC-V architectural rule.

Rewrite it into one clearer, more actionable, directly testable rule for assembly test generation.

Output only the rewritten rule. No explanation. No bullet. No numbering.

Requirements:
- Keep the original meaning.
- Make the rule observable through architectural state when possible.
- Prefer violation-based cause-effect wording:
  "If <required condition is violated>, then <observable architectural result>."
- Do not output rules in the form "<thing> must be <constraint>" unless the observable consequence is also stated.
- For constraint rules, explicitly state what happens when the constraint is violated.
- If violating a constraint causes an exception, explicitly mention the exception.
- Prefer concrete observable outcomes such as "raises an illegal-instruction exception", "sets mcause to ...", "the CSR readback value is ...", or "the destination register contains ...".
- Make implicit constraints explicit.
- If the old rule is already clear and directly testable, keep it mostly unchanged.
- If the old rule only states metadata or a constraint, rewrite it into the architectural consequence of violating or satisfying that constraint.
- Do not invent unrelated behavior.
- Do not make the rule more specific than the old rule unless needed to make it testable.

Old rule:
{STRING_RULE}'''

    STRING_RESPONSE = FUNCTION_CALL_LLM(STRING_PROMPT)
    STRING_RESPONSE = STRING_RESPONSE.strip()

    STRING_RESPONSE = re.sub(r'^```(?:\w+)?\s*', '', STRING_RESPONSE)
    STRING_RESPONSE = re.sub(r'\s*```$', '', STRING_RESPONSE)
    STRING_RESPONSE = STRING_RESPONSE.strip()

    STRING_RESPONSE = re.sub(r'(?i)^RULE\s+\d+\s*:\s*', '', STRING_RESPONSE).strip()

    return STRING_RESPONSE

LIST_REWRITTEN_OUTPUT_LINE = []

for EACH_LINE in STRING_OUTPUT.splitlines(keepends=True):
    STRING_LINE = EACH_LINE.rstrip('\n')

    MATCH_RULE = re.match(r'^(RULE\s+\d+:\s*)(.+?)\s*$', STRING_LINE)

    if not MATCH_RULE:
        LIST_REWRITTEN_OUTPUT_LINE.append(EACH_LINE)
        continue

    STRING_RULE_PREFIX = MATCH_RULE.group(1)
    STRING_OLD_RULE = MATCH_RULE.group(2).strip()

    IS_TESTABLE = FUNCTION_CHECK_RULE_TESTABLE(STRING_OLD_RULE)

    if IS_TESTABLE:
        LIST_REWRITTEN_OUTPUT_LINE.append(EACH_LINE)
        continue

    STRING_NEW_RULE = FUNCTION_REWRITE_RULE_TESTABLE(STRING_OLD_RULE)

    if not STRING_NEW_RULE:
        STRING_NEW_RULE = STRING_OLD_RULE

    LIST_REWRITTEN_OUTPUT_LINE.append(f'{STRING_RULE_PREFIX}{STRING_NEW_RULE}\n')

STRING_OUTPUT = ''.join(LIST_REWRITTEN_OUTPUT_LINE).strip() + '\n'

def FUNCTION_REMOVE_EMPTY_INSTRUCTION_BLOCKS(STRING_TEXT):
    LIST_LINE = STRING_TEXT.splitlines(keepends=True)
    LIST_BLOCK = []
    LIST_OUTPUT_BLOCK = []

    def FUNCTION_FLUSH_BLOCK():
        nonlocal LIST_BLOCK

        if not LIST_BLOCK:
            return

        HAS_INSTRUCTION = any(EACH_LINE.strip().startswith('INSTRUCTION:') for EACH_LINE in LIST_BLOCK)
        HAS_RULE = any(re.match(r'^\s*RULE\s+\d+\s*:', EACH_LINE) for EACH_LINE in LIST_BLOCK)

        if (not HAS_INSTRUCTION) or HAS_RULE:
            while LIST_BLOCK and not LIST_BLOCK[0].strip():
                LIST_BLOCK.pop(0)
            while LIST_BLOCK and not LIST_BLOCK[-1].strip():
                LIST_BLOCK.pop()

            if LIST_BLOCK:
                if LIST_OUTPUT_BLOCK:
                    LIST_OUTPUT_BLOCK.append('\n')
                LIST_OUTPUT_BLOCK.extend(LIST_BLOCK)

        LIST_BLOCK = []

    for EACH_LINE in LIST_LINE:
        if EACH_LINE.strip().startswith('INSTRUCTION:'):
            FUNCTION_FLUSH_BLOCK()
            LIST_BLOCK = [EACH_LINE]
        else:
            LIST_BLOCK.append(EACH_LINE)

    FUNCTION_FLUSH_BLOCK()

    return ''.join(LIST_OUTPUT_BLOCK).strip() + '\n'

STRING_OUTPUT = FUNCTION_REMOVE_EMPTY_INSTRUCTION_BLOCKS(STRING_OUTPUT)

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_OUTPUT)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 5 - CREATE - 5_INSTRUCTION_FINAL.txt - END

4 minute(s) 38.30 second(s)

====
DONE
====


In [8]:
# 6 - CREATE - 6_INSTRUCTION_TOTAL.txt - START

"""
Maintain a cumulative collection of RISC-V instructions, assembly formats, and architectural rules by merging new results, replacing outdated entries, and sorting instructions alphabetically.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import os, re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '4_INSTRUCTION/1_OUTPUT/5_INSTRUCTION_FINAL.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/6_INSTRUCTION_TOTAL.txt'
# PATH - FINISH

# VARIABLE - BEGIN
IS_NEED_INIT = (not os.path.exists(FILE_OUTPUT)) or os.path.getsize(FILE_OUTPUT) == 0
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    TEXT_INPUT = FILE_HANDLE.read()

if IS_NEED_INIT:
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(TEXT_INPUT)
else:
    with open(FILE_OUTPUT, encoding='utf-8') as FILE_HANDLE:
        TEXT_OUTPUT = FILE_HANDLE.read()
    TEXT_COMBINED = TEXT_OUTPUT.strip() + '\n\n' + TEXT_INPUT.strip()
    REGEX_INSTRUCTION = re.compile(r'(?ms)^\s*INSTRUCTION:\s*([^\n]+)\s*\n.*?(?=^\s*INSTRUCTION:\s*[^\n]+\s*\n|\Z)')
    MAP_BY_NAME = {EACH_MATCH.group(1).strip(): EACH_MATCH.group(0) for EACH_MATCH in REGEX_INSTRUCTION.finditer(TEXT_COMBINED)}
    TEXT_FINAL = '\n\n'.join(MAP_BY_NAME[EACH_KEY].strip() for EACH_KEY in sorted(MAP_BY_NAME, key=str.upper)).strip()
    with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
        FILE_HANDLE.write(TEXT_FINAL)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 6 - CREATE - 6_INSTRUCTION_TOTAL.txt - END

0.00 second(s)

====
DONE
====


In [9]:
# 7 - CREATE - 7_INSTRUCTION_SUMMARY.txt - START

"""
Generate concise summaries of RISC-V instruction semantics from extracted architectural rules using LLMs, with chunk-based summarization and merging for large rule sets.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from ollama import chat, Client
from typing import Dict, List, Optional, Tuple
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '4_INSTRUCTION/1_OUTPUT/6_INSTRUCTION_TOTAL.txt'
FILE_OUTPUT = '4_INSTRUCTION/1_OUTPUT/7_INSTRUCTION_SUMMARY.txt'
FILE_PROMPT_1 = '4_INSTRUCTION/2_HISTORY/4_HISTORY_INSTRUCTION_SUMMARY_1.txt'
FILE_PROMPT_2 = '4_INSTRUCTION/2_HISTORY/5_HISTORY_INSTRUCTION_SUMMARY_2.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_SANITIZE_TEXT(STRING_VALUE: str) -> str:
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_RANGE_VALUE in range(0xe000, 0xf900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_RANGE_VALUE), '')
    return (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+').replace("’", "'").replace("‘", "'").strip())

def FUNCTION_CALL_LLM(STRING_PROMPT: str) -> Tuple[str, str]:
    if IS_LOCAL:
        STRING_RAW = chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content
    else:
        STRING_RAW = CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content
    return FUNCTION_SANITIZE_TEXT(STRING_RAW), STRING_RAW

def FUNCTION_ADD_HISTORY(WHICH: int, GLOBAL_NUMBER: int, LOCAL_NUMBER: int, STRING_NAME: str, STRING_PROMPT: str, STRING_RAW_RESPONSE: str):
    INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH, '=')
    INTRO = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * HALF_WIDTH}
=======
PROMPT:
=======
{'=' * HALF_WIDTH}
'''
    ENDING = f'''
{'=' * HALF_WIDTH}
=========
RESPONSE:
=========
{'=' * HALF_WIDTH}
{STRING_RAW_RESPONSE}
'''
    return INTRO_FIRST + '\n' + INTRO + STRING_PROMPT + ENDING + INTRO_END + SUFFIX

def FUNCTION_PARSE_INSTRUCTION_RULE(TEXT_VALUE: str) -> Dict[str, List[Tuple[int, str]]]:
    INSTRUCTION_CURRENT = None
    MAP_INSTRUCTION = {}
    for EACH_LINE in TEXT_VALUE.splitlines():
        MATCH_INSTRUCTION = REGEX_INSTRUCTION.match(EACH_LINE)
        if MATCH_INSTRUCTION:
            INSTRUCTION_CURRENT = MATCH_INSTRUCTION.group(1).strip()
            MAP_INSTRUCTION.setdefault(INSTRUCTION_CURRENT, [])
            continue
        MATCH_RULE = REGEX_RULE.match(EACH_LINE)
        if MATCH_RULE and INSTRUCTION_CURRENT is not None:
            MAP_INSTRUCTION[INSTRUCTION_CURRENT].append((int(MATCH_RULE.group(1)), MATCH_RULE.group(2).strip()))
    for EACH_INSTRUCTION_NAME in MAP_INSTRUCTION:
        MAP_INSTRUCTION[EACH_INSTRUCTION_NAME].sort(key=lambda EACH_X: EACH_X[0])
    return MAP_INSTRUCTION

def FUNCTION_CHUNK_RULE(LIST_RULE: List[Tuple[int, str]], CHUNK_COUNT: int):
    return [LIST_RULE[EACH_INDEX:EACH_INDEX + CHUNK_COUNT] for EACH_INDEX in range(0, len(LIST_RULE), CHUNK_COUNT)]

def FUNCTION_FORMAT_RULE(LIST_RULE_CHUNK: List[Tuple[int, str]]) -> str:
    return '\n'.join(f'RULE {EACH_NUMBER_RULE}: {EACH_TEXT_RULE}' for EACH_NUMBER_RULE, EACH_TEXT_RULE in LIST_RULE_CHUNK)

def FUNCTION_SUMMARIZE_INSTRUCTION(INSTRUCTION_NAME: str, LIST_RULE: List[Tuple[int, str]]):
    global STRING_PROMPT_HISTORY_1, GLOBAL_COUNTER_1, LOCAL_COUNTER_1
    global STRING_PROMPT_HISTORY_2, GLOBAL_COUNTER_2, LOCAL_COUNTER_2
    LIST_PARTIAL = []
    for EACH_INDEX_CHUNK, EACH_LIST_CHUNK in enumerate(FUNCTION_CHUNK_RULE(LIST_RULE, CHUNK_SIZE), 1):
        STRING_PROMPT = STRING_PROMPT_TEMPLATE_CHUNK.format(instruction=INSTRUCTION_NAME, rules_text=FUNCTION_FORMAT_RULE(EACH_LIST_CHUNK))
        STRING_CLEANED, STRING_RAW = FUNCTION_CALL_LLM(STRING_PROMPT)
        STRING_PROMPT_HISTORY_1 += FUNCTION_ADD_HISTORY(1, GLOBAL_COUNTER_1, LOCAL_COUNTER_1, INSTRUCTION_NAME, STRING_PROMPT, STRING_RAW)
        GLOBAL_COUNTER_1 += 1
        LOCAL_COUNTER_1 += 1
        LIST_PARTIAL.append(STRING_CLEANED)
    if len(LIST_PARTIAL) == 1:
        return LIST_PARTIAL[0]
    STRING_PROMPT_MERGE = STRING_PROMPT_TEMPLATE_MERGE.format(instruction=INSTRUCTION_NAME, partials='\n\n'.join(f'- {EACH_PARTIAL}' for EACH_PARTIAL in LIST_PARTIAL))
    STRING_CLEANED, STRING_RAW = FUNCTION_CALL_LLM(STRING_PROMPT_MERGE)
    STRING_PROMPT_HISTORY_2 += FUNCTION_ADD_HISTORY(2, GLOBAL_COUNTER_2, LOCAL_COUNTER_2, INSTRUCTION_NAME, STRING_PROMPT_MERGE, STRING_RAW)
    GLOBAL_COUNTER_2 += 1
    LOCAL_COUNTER_2 += 1
    return STRING_CLEANED
# FUNCTION - FINISH

# VARIABLE - BEGIN
CHUNK_SIZE = 10000
REGEX_INSTRUCTION = re.compile(r'^\s*INSTRUCTION:\s*(.+?)\s*$')
REGEX_RULE = re.compile(r'^\s*RULE\s+(\d+)\s*:\s*(.+?)\s*$')
DIVIDER_LINE = (('#' * WIDTH + '\n') * 3).rstrip('\n')
SUFFIX = '\n' + DIVIDER_LINE + '\n'
HALF_WIDTH = WIDTH // 2
STRING_PROMPT_HISTORY_1 = ''
STRING_PROMPT_HISTORY_2 = ''
GLOBAL_COUNTER_1 = 1
LOCAL_COUNTER_1 = 1
GLOBAL_COUNTER_2 = 1
LOCAL_COUNTER_2 = 1
LIST_OUTPUT_LINE: List[str] = []
INSTRUCTION_INITIAL: Optional[str] = None
STRING_PROMPT_TEMPLATE_CHUNK = '''You are summarizing RISC-V instruction semantics strictly from the provided rules.

Instruction: {instruction}

Rules (subset):
{rules_text}

Task:
Write a concise technical summary (maximum 4 sentences) capturing only the high-level semantics explicitly stated in the rules.

Focus on:
- register purpose
- key fields (group similar fields together)
- access control and privilege restrictions
- notable exception behavior

Rules:
- Do NOT list every field bit-by-bit.
- Do NOT enumerate rule numbers.
- Do NOT include redundant or repetitive details.
- Group related behaviors into one sentence.
- Omit low-level details (e.g., exact bit positions, aliases, mirrored bits) unless they are essential.
- Prefer clarity over completeness.

Output must be ASCII-only text.
Return only the paragraph.'''
STRING_PROMPT_TEMPLATE_MERGE = '''You are merging partial summaries of the same RISC-V instruction derived from rule subsets.

Instruction: {instruction}

Partial summaries:
{partials}

Task:
Produce a single cohesive summary (4-6 sentences, one paragraph) that preserves all semantics present in the partial summaries.
- Do NOT introduce new semantics not present in the partial summaries.
- Do NOT remove constraints, corner cases, or conditions.
- Remove redundancy by merging equivalent statements only.
- Prefer canonical RISC-V wording and precise terminology.
Remember, output must be ASCII-only text.
Return only the final paragraph.'''
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT, encoding='utf-8') as FILE_HANDLE:
    MAP_INSTRUCTION = FUNCTION_PARSE_INSTRUCTION_RULE(FILE_HANDLE.read())

for EACH_INDEX_ITEM, (EACH_INSTRUCTION_NAME, EACH_LIST_RULE) in enumerate(MAP_INSTRUCTION.items()):
    if EACH_INDEX_ITEM == 0:
        INSTRUCTION_INITIAL = EACH_INSTRUCTION_NAME
    elif EACH_INSTRUCTION_NAME != INSTRUCTION_INITIAL:
        INSTRUCTION_INITIAL = EACH_INSTRUCTION_NAME
        LOCAL_COUNTER_1 = 1
        LOCAL_COUNTER_2 = 1
    STRING_SUMMARY = FUNCTION_SUMMARIZE_INSTRUCTION(EACH_INSTRUCTION_NAME, EACH_LIST_RULE)
    LIST_OUTPUT_LINE += [f'INSTRUCTION: {EACH_INSTRUCTION_NAME}', f'SUMMARY: {STRING_SUMMARY}', '']

with open(FILE_PROMPT_1, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_PROMPT_HISTORY_1.removesuffix(SUFFIX))

with open(FILE_PROMPT_2, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write(STRING_PROMPT_HISTORY_2.removesuffix(SUFFIX))

with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    FILE_HANDLE.write('\n'.join(LIST_OUTPUT_LINE).strip())
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 7 - CREATE - 7_INSTRUCTION_SUMMARY.txt - END

13.43 second(s)

====
DONE
====
